# Galaxy10 DECaLS inspection

This notebook inspects the cached HDF5 file and the deterministic split manifest. It does not modify the dataset. Run `python scripts/prepare_galaxy10.py` first.

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from galaxy_classification.galaxy10 import (
    GALAXY10_CLASS_NAMES, GALAXY10_FILENAME, GALAXY10_SHAPE, validate_manifest
)

H5_PATH = Path.home() / '.astroNN' / 'datasets' / GALAXY10_FILENAME
MANIFEST_PATH = ROOT / 'data' / 'image-manifests' / 'galaxy10-decals.csv'
manifest = pd.read_csv(MANIFEST_PATH)
validate_manifest(manifest, GALAXY10_SHAPE[0])
H5_PATH, MANIFEST_PATH

## Dataset structure

Galaxy10 DECaLS contains 17,736 display images built from the g, r, and z survey bands. Labels are Galaxy Zoo consensus-derived morphology classes.

In [ ]:
with h5py.File(H5_PATH, 'r') as dataset:
    structure = {key: dataset[key].shape for key in dataset.keys()}
structure

In [ ]:
class_distribution = (
    manifest.groupby(['label', 'class_name']).size().rename('images').reset_index()
)
split_distribution = manifest.groupby(['split', 'label']).size().unstack(fill_value=0)
display(class_distribution)
display(split_distribution)

## Representative examples

In [ ]:
figure, axes = plt.subplots(10, 3, figsize=(9, 27))
with h5py.File(H5_PATH, 'r') as dataset:
    images = dataset['images']
    for label, class_name in enumerate(GALAXY10_CLASS_NAMES):
        sample_indices = manifest.loc[manifest['label'].eq(label), 'sample_index'].head(3)
        for column, sample_index in enumerate(sample_indices):
            axes[label, column].imshow(images[int(sample_index)])
            axes[label, column].axis('off')
            if column == 0:
                axes[label, column].set_title(f'{label}: {class_name}', loc='left')
figure.tight_layout()

## Channel-value inspection

In [ ]:
rng = np.random.default_rng(42)
sample_indices = rng.choice(GALAXY10_SHAPE[0], size=500, replace=False)
with h5py.File(H5_PATH, 'r') as dataset:
    sample = dataset['images'][np.sort(sample_indices)].astype(np.float32) / 255.0
channel_summary = pd.DataFrame({
    'channel': ['g-derived', 'r-derived', 'z-derived'],
    'mean': sample.mean(axis=(0, 1, 2)),
    'std': sample.std(axis=(0, 1, 2)),
    'minimum': sample.min(axis=(0, 1, 2)),
    'maximum': sample.max(axis=(0, 1, 2)),
})
channel_summary